# GEMINI Sense Annotation Pipeline (Refactored)
This notebook uses the new modular pipeline for sense annotation with Gemini, leveraging shared utilities for maintainability and traceability.

In [1]:
from config import OPENAI_API_KEY, GOOGLE_GENAI_API_KEY, SENSE_REPO, ANNOTATIONS_TSV, OUTPUT_DIR
from process_senses import process_senses_with_chain
from writers import CustomWebAnnoTSVWriter, IncetprionWebAnnoTSVWriter
from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
import pandas as pd
import time

#decide which model to use from
model="gemini-2.0-flash-lite" 

e:\Github\LexiSense-SR\writers.py:77: SyntaxWarning: invalid escape sequence '\_'
  code = code.replace('/', '\_')


In [ ]:
# Load sense repository and annotated sentences
from data_loader import load_sense_repo
senses_df = load_sense_repo()
parser = WebAnnoLEXISParser(ANNOTATIONS_TSV)
sentences = parser.parse()

# Define test range (adjust as needed)
start_sentence = 0
end_sentence = start_sentence + 100
sentences = sentences[start_sentence:end_sentence]

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate

# Compose Gemini LLM chain
system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:
{{
  "sense_id": "<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
}}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
"""
user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...<\\/b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", user_prompt)
])

llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_GENAI_API_KEY,
    model=model,
)
# Set model origin for traceability
ORIGIN_LLM = f"Gemini_{model}"

parser = StrOutputParser()
chain = prompt | llm | parser

In [4]:
# Annotate sentences using the shared pipeline
start_time = time.time()
annotated_sentences = process_senses_with_chain(
    sentences,
    senses_df,
    chain,
    ORIGIN_LLM,
    time_delay=5.0
)
end_time = time.time()
print(f"Processed sentences {start_sentence} to {end_sentence} in {end_time - start_time:.2f} seconds.")

Processed 100/100 sentences.Processed sentences 0 to 100 in 4607.71 seconds.


In [5]:
# Write debug output (all layers, including AI notes and candidates)
debug_writer = CustomWebAnnoTSVWriter(annotated_sentences)
debug_writer.save(OUTPUT_DIR / f"gemini_{model}_debug_{start_sentence+1:04d}_{end_sentence:04d}.tsv")

In [6]:
# Write Inception-compatible output (for annotation import)
incept_writer = IncetprionWebAnnoTSVWriter(annotated_sentences)
incept_writer.save(OUTPUT_DIR / f"gemini_inception_{model}_{start_sentence+1:04d}_{end_sentence:04d}.tsv")

## Log processing time and completion

In [7]:
with open("gemini.log", "a", encoding="utf-8") as f:
    f.write(f"Processed sentences {start_sentence} to {end_sentence}\n")
    f.write(f"Gemini {model} took {end_time - start_time:.2f} seconds\n")
    f.write(f"Or minutes: {(end_time - start_time)/60:.2f}\n")
print("Done.")

Done.
